# పాఠం 01 - AI ఏజెంట్‌లకు పరిచయం

**బెగినర్ల కోసం AI ఏజెంట్లు** కోర్సులోని మొదటి పాఠానికి స్వాగతం!

ఒక **AI ఏజెంట్** అనేది పెద్ద భాషా మోడల్ (LLM)ని దాని తర్క మోటార్‌గా ఉపయోగించే ప్రోగ్రామ్ మరియు వాస్తవ ప్రపంచంలో *చర్యలు* తీసుకోవచ్చు — APIలను కాల్ చేయడం, డేటాబేస్‌లను ప్రశ్నించడం లేదా కోడ్ నడపడం ద్వారా — యూజర్‌ తరఫున లక్ష్యాన్ని సాధించడానికి.

ఈ నోట్‌బుక్‌లో మీరు మీ మొదటి ఏజెంట్‌ను నిర్మిస్తారు: సముద్ర తీర గమ్యస్థానాలను సిఫార్సు చేసే **ట్రావెల్ ఏజెంట్**. ఈ ప్రయాణంలో మీరు నేర్చుకోబోతున్నారు ఎలా:

1. **Microsoft Agent Framework** ఉపయోగించి Microsoft Foundry Agent Service కు కనెక్ట్ కావడం.
2. ఏజెంట్‌కు ఒక **టూల్** ఇవ్వడం — అది పిలవగల సాధారణ Python ఫంక్షన్.
3. ఏజెంట్‌ను నడపడం మరియు దాని ప్రతిస్పందనని పరిశీలించడం.
4. ఏజెంట్ యొక్క ప్రతిస్పందనను టోకెన్-టోకెన్‌గా స్ట్రీమ్ చేయడం.


## సెటప్

ఈ నోట్‌బుక్ ను రన్ చేయడానికి ముందు, మీరు కలిగి ఉండాలి:

1. **మైక్రోసాఫ్ట్ foundry ప్రాజెక్ట్** ఒక డిప్లాయ్ చేసిన చాట్ మోడెల్ తో (ఉదా: `gpt-5-mini`).
2. **Azure CLI లో లాగిన్ అయి ఉండాలి** — మీ టెర్మినల్ లో `az login` ను రన్ చేయండి.
3. **ఆవశ్యకమైన ఎన్విరాన్‌మెంట్ వేరియబుల్స్ సెట్ చేసుకోండి:**
   - `AZURE_AI_PROJECT_ENDPOINT` — మీ మైక్రోసాఫ్ట్ foundry ప్రాజెక్ట్ ఎండ్పాయింట్.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — మీరు డిప్లాయ్ చేసిన మోడల్ పేరు.

క్రింద ఉన్న సెల్ మీరు అవసరమయ్యే Python ప్యాకేజీలను ఇన్‌స్టాల్ చేస్తుంది.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## మీ మొదటి ఏజెంట్‌ను సృష్టించడం

ఒక ఏజెంటుకు రెండు విషయాలు అవసరం:

- **సూచనలు** ఇది *ఎవరు* మరియు *ఎలా వ్యవహరించాలి* అని తెలియజేస్తాయి (ఒక సిస్టమ్ ప్రాంప్ట్).
- **పరికరాలు** — ఏజెంట్ సమాచారాన్ని పొందడానికి లేదా క్రియలు చేయడానికి పిలవగల `@tool` తో అలంకరించబడిన Python ఫంక్షన్లు.

కింద ఒక సరళమైన పరికరం నిర్వచించబడింది ఇది ప్రాచుర్యమైన విశ్రాంతి గమ్యస్థలాల జాబితాను ఇస్తుంది. యూజర్ ప్రయాణ సిఫార్సులు అడిగినప్పుడు ఏజెంట్ ఈ పరికరాన్ని ఉపయోగిస్తుంది.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## స్ట్రీమింగ్ స్పందనలు

మరింత పరస్పర చర్య అనుభవాన్ని కోసం మీరు ఏజెంట్ యొక్క స్పందనను **స్ట్రీమ్** చేయవచ్చు. పూర్తి స్పందన కోసం ఎదురుచూసే బదులు, ఏజెంట్ ఉత్పత్తి చేసే టెక్స్ట్ భాగాలను వెంటనే అందిస్తుంది. ఇది ముఖ్యంగా మీరు రియల్ టైంలో అవుట్పుట్ చూపించాలనుకునే చాట్ ఇంటర్‌ఫేస్‌లలో చాలా ఉపయోగకరం.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## సారం

ఈ పాఠంలో మీరు ఎలా నేర్చుకున్నారు:

- **ప్రొవైడర్‌ను సృష్టించండి** ఇది Microsoft Foundry Agent Service కు `FoundryChatClient` ద్వారా కనెక్ట్ అవుతుంది.
- **@tool డెకరేటర్ ఉపయోగించి టూల్ నిర్వచించండి** తద్వారా ఏజెంట్ మీ Python ఫంక్షన్లను పిలవగలదు.
- **ఉపయోగించేవారు సందేశంతో ఏజెంట్‌ను నడపండి** మరియు దాని ప్రతిస్పందనను ప్రింట్ చేయండి.
- **ప్రతిస్పందనలను స్ట్రీమ్ చేయండి** రియల్-టైమ్ అవుట్పుట్ కోసం.

తదుపరి పాఠంలో మేము ఏజెంటిక్ ఫ్రేమ్‌వర్క్‌లను మరింత లోతుగా పరిశీలించి ఏజెంట్లకు మరింత శక్తివంతమైన టూల్స్ మరియు బహుళ దశల తర్క శక్తులను ఇచ్చే విధానం నేర్చుకుంటాము.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్వీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలు తప్పులు లేదా అసమగ్రతలను కలిగి ఉండవచ్చు. దాని స్వదేశ భాషలో ఉన్న అసలు పత్రాన్ని అధికారం కలిగిన మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదారులు కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
